In [3]:
import os
import sys

import torch
import yaml
from transformers import TrainingArguments

PROJECT_ROOT = os.path.abspath(
    os.path.join(os.getcwd(), "..")
    if os.path.basename(os.getcwd()) == "notebooks"
    else os.getcwd()
)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from srcs.datasets.vicocktail import Collator, load_vicocktail
from srcs.nets.e2e import get_model as create_model
from srcs.nets.utils import ctc_decode, load_weights
from srcs.spm.spm_train import ensure_unigram
from srcs.spm.text_transofm import TextTransform
from srcs.trainer.trainer import (
    HFTrainer,
    build_metric_fn,
    preprocess_logits_for_metrics,
)

CONFIG_PATH = os.path.join(PROJECT_ROOT, "config.yaml")
CHECKPOINT_PATH = os.path.join(
    PROJECT_ROOT, "checkpoints", "finetune_vsr_12", "final"
)
TEST_FRACTION = 1.0
SAMPLE_COUNT = 3

In [2]:
with open(CONFIG_PATH, encoding="utf-8") as file:
    config = yaml.safe_load(file)

evaluation_config = config["evaluation"]
test_dataset = load_vicocktail(
    test_fraction=TEST_FRACTION,
    splits=("test",),
)["test"]

model_path, units_path = ensure_unigram()
text_transform = TextTransform(model_path, units_path)
model = create_model("auto-vsr", text_transform.vocab_size, size="large")
load_weights(model, CHECKPOINT_PATH)

test_collator = Collator(text_transform, "test")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Test samples: {len(test_dataset)}")

Checkpoint: d:\projects\VietnameseVSR\checkpoints\finetune_vsr_12\final
Test samples: 1167


In [3]:
evaluation_args = TrainingArguments(
    output_dir=os.path.join(PROJECT_ROOT, "checkpoints"),
    label_names=["labels", "label_lengths"],
    per_device_eval_batch_size=evaluation_config["batch_size"],
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    dataloader_num_workers=evaluation_config["num_workers"],
    dataloader_pin_memory=torch.cuda.is_available(),
    dataloader_persistent_workers=evaluation_config["num_workers"] > 0,
    dataloader_prefetch_factor=(
        2 if evaluation_config["num_workers"] > 0 else None
    ),
)

trainer = HFTrainer(
    model=model,
    args=evaluation_args,
    train_dataset=None,
    eval_dataset=test_dataset,
    train_collator=test_collator,
    validation_collator=test_collator,
    compute_metrics=build_metric_fn(text_transform),
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
)

output = trainer.predict(test_dataset)
trainer.log_metrics("test", output.metrics)

***** test metrics *****
  test_loss                   =    61.4843
  test_model_preparation_time =      0.003
  test_runtime                = 0:01:00.34
  test_samples_per_second     =     19.339
  test_steps_per_second       =      2.419
  test_wer                    =     0.4838


In [4]:
frame_ids, input_lengths = output.predictions
labels, label_lengths = output.label_ids
token_ids = ctc_decode(
    torch.as_tensor(frame_ids),
    torch.as_tensor(input_lengths),
    text_transform.blank_id,
)

for index in range(min(SAMPLE_COUNT, len(token_ids))):
    prediction = text_transform.decode(token_ids[index])
    reference = text_transform.decode(labels[index][: int(label_lengths[index])])

    print(f"Sample {index + 1}")
    print(f"Reference:  {reference}")
    print(f"Prediction: {prediction}")
    print()

Sample 1
Reference:  khi mà mình rõ được những cái điều mà mình ưu tiên trong cuộc sống thì mọi lựa chọn chúng mình đưa ra đều lấy nó làm kim chỉ nam để xác định là nên nói có hay là nó không học cách
Prediction: khi mà mình rõ được những cái điều mà mình mới ưu tiên cho cuộc sống cái mọi lựa con của mình đưa ra đều này đó làm gì nghĩ làm để xác định ra này hỏi rồi không học cách

Sample 2
Reference:  mười bài học mà mình học được trong năm nay mình đã chọn lọc ra và sẽ chia sẻ với bạn
Prediction: tốt được bài hồng mà mình không cộng nào này mình đã chọn không nào định trên thì sẻ với bài

Sample 3
Reference:  dơ sai ừn ợp seo lơn ninh khoa học về việc tự học trong cuốn sách này ngoài việc đề cập tới việc tự học thì nó cũng đề cập tới việc là
Prediction: trắng học seo với khoa học về việc tự học thông cũng sách là hóa về những gặp tới việc tự học thì cũng biết cảm thấy việc là



## Refiner mask checks

This section uses the actual `VisualRefiner` implementation with a small synthetic batch so padding and the local attention window are easy to inspect.

In [ ]:
import torch
import matplotlib.pyplot as plt

from srcs.nets.backend.nets_utils import make_non_pad_mask
from srcs.nets.backend.refiner.refiner import VisualRefiner

torch.manual_seed(42)

batch_size = 2
time_steps = 8
vocab_size = 12
visual_dim = 8
attn_dim = 16
attn_heads = 4
window_size = 3
test_lengths = torch.tensor([8, 5])

refiner = VisualRefiner(
    vocab_size=vocab_size,
    visual_dim=visual_dim,
    attn_dim=attn_dim,
    attn_head=attn_heads,
    ffn_dim=32,
    num_blocks=1,
    window_size=window_size,
    attn_dropout=0.0,
    dropout=0.0,
).eval()

test_logits = torch.randn(batch_size, time_steps, vocab_size)
test_visual = torch.randn(batch_size, time_steps, visual_dim)

with torch.no_grad():
    refiner_output = refiner(test_logits, test_visual, test_lengths)

valid_mask = make_non_pad_mask(test_lengths).bool()
padding_mask = ~valid_mask
local_mask = refiner.make_local_mask(time_steps, valid_mask.device)
cross_mask = local_mask.unsqueeze(0) | padding_mask.unsqueeze(1)

positions = torch.arange(time_steps)
expected_local_mask = (
    positions[:, None] - positions[None, :]
).abs() > window_size // 2

assert padding_mask.shape == (batch_size, time_steps)
assert local_mask.shape == (time_steps, time_steps)
assert cross_mask.shape == (batch_size, time_steps, time_steps)
assert torch.equal(padding_mask, ~valid_mask)
assert torch.equal(local_mask, expected_local_mask)
assert torch.isfinite(refiner_output["final_logits"]).all()
assert torch.isfinite(refiner_output["inner_logits"]).all()

padded_hidden = refiner_output["refiner_hidden"].masked_select(
    padding_mask.unsqueeze(-1).expand_as(refiner_output["refiner_hidden"])
)
assert padded_hidden.abs().max().item() == 0.0

print(f"valid_mask: {tuple(valid_mask.shape)}")
print(f"padding_mask: {tuple(padding_mask.shape)}")
print(f"local_mask: {tuple(local_mask.shape)}")
print(f"cross_mask: {tuple(cross_mask.shape)}")
print(f"Maximum padded hidden: {padded_hidden.abs().max().item()}")
print("All mask checks passed.")

In [ ]:
sample_index = 1
padding_mask_matrix = padding_mask[sample_index].unsqueeze(0).expand(
    time_steps, -1
)

figure, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(
    valid_mask[sample_index].unsqueeze(0).cpu(),
    cmap="Greys",
    vmin=0,
    vmax=1,
    aspect="auto",
)
axes[0].set_title("Valid mask [T]")
axes[0].set_xlabel("Key frame")
axes[0].set_yticks([])

axes[1].imshow(
    padding_mask_matrix.cpu(),
    cmap="Greys",
    vmin=0,
    vmax=1,
    interpolation="nearest",
)
axes[1].set_title("Self key padding mask (True = blocked)")
axes[1].set_xlabel("Key frame")
axes[1].set_ylabel("Query frame")

axes[2].imshow(
    cross_mask[sample_index].cpu(),
    cmap="Greys",
    vmin=0,
    vmax=1,
    interpolation="nearest",
)
axes[2].set_title("Local cross mask (True = blocked)")
axes[2].set_xlabel("Key frame")
axes[2].set_ylabel("Query frame")

figure.suptitle(
    f"Sample {sample_index}: length={test_lengths[sample_index].item()}, "
    f"window_size={window_size}"
)
plt.tight_layout()
plt.show()